# CA simulation: linear DGP and Predictive R²

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from pathlib import Path

!pip install tables


from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_PREDICTIVE_with_x_t')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)



# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration


####################################
## DEFINE DATA GENERATING PROCESS ##
####################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1  # subtract the last time period because the characteristics are lagged test size defines the size of the index of the returns.

    train_data = r_adjusted[:, :train_size]  # Training data up to train_size
    val_data = r_adjusted[:, train_size:train_size + val_size]  # Validation data from train_size to train_size + val_size
    test_data = r_adjusted[:, train_size + val_size:]  # Testing data from train_size + val_size to the end

    print(train_data.shape)
    print(val_data.shape)
    print(test_data.shape)

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size


###############################################################################
## Define CA model with a dynamic number of layers that can be iterated over ##
###############################################################################


def make_model(N, Pc, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    x = input_beta

    if num_hidden_layers >= 1:
        # First hidden layer with 32 neurons and batch normalization
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    if num_hidden_layers >= 2:
        # Second hidden layer with 16 neurons and batch normalization
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    if num_hidden_layers >= 3:
        # Third hidden layer with 8 neurons and batch normalization
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

###########################################################################
## DEFINE HELPER FUNCTIONS FOR THE TRAINING LOOP TO MAKE IT MORE COMPACT ##
###########################################################################

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

# Function for calculation of the sharpe ratio
def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

# Function to calculate the long-short portfolio
def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    # Number of assets
    N = predicted_returns.shape[0]

    # Determine the threshold for the top and bottom deciles
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

    # Identify the long and short positions
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold

    # Calculate the returns for the long and short positions
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)

    # Calculate the portfolio return (long returns - short returns)
    portfolio_return = long_returns - short_returns

    return portfolio_return

#######################################
## DEFINE PARAMETERS TO ITERATE OVER ##
#######################################

# Define the different model configurations
models = [0, 1, 2, 3]  # Number of hidden layers
#
# Monte Carlo Simulation with L1 tuning and Early Stopping
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]  # Example factor list
#
# L1 regularization values to search over
l1_values = [0.00001, 0.001]
#

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


# Define columns
cols = ['Model', 'Factors', '$x_t$ Sharpe Ratio', 'Long-Short Portfolio SR', '$R^2$ for $r_t$', 'Best L_1 Value', 'Adjusted $R^2$ DGP', '$R^2$ for $x_t$']

#######################################
## training loop with simulated data ##
#######################################

results = []  # Define results here
scores_dict = {} # Define scores_dict here
for num_hidden_layers in tqdm(models):
    for factor in tqdm(factors):
        scores = []
        for i in tqdm(range(num_simulations)):
            train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

            print(train_data.shape)
            print(val_data.shape)
            print(test_data.shape)

            ################
            ## TRAIN DATA ##
            ################


            # Prepare inputs for the factor network input calculation
            input_beta_train = np.array([c_normalized[:, :, t] for t in range(train_size)])
            input_beta_train_aligned = input_beta_train[:-1, :, :]  # Cut off the last observation, shape (107, 200, 50)

            # Calculate forward returns for training data
            r_t_train = np.array([train_data[:, t+1] for t in range(train_size - 1)])  # Forward returns


            # Calculate the input for the factor network
            input_factor_train = calculate_factor_network_input(input_beta_train_aligned, r_t_train)

            # Ensure the shape is consistent
            assert input_beta_train_aligned.shape[0] == r_t_train.shape[0], "Shape mismatch in training data"

            ######################
            ### VALIDATION DATA ##
            ######################

            input_beta_val = np.array([c_normalized[:, :, t] for t in range(train_size, train_size + val_size)])  # Shift the returns forward
            input_beta_val_aligned = input_beta_val[:-1, :, :]  # Align correctly

            # r_t_val = np.array([val_data[:, t] for t in range(1, val_size + 1)])  # Shift the returns forward

            # Calculate forward returns for training data
            r_t_val = np.array([val_data[:, t+1] for t in range(val_size - 1)])  # Forward returns
            print(f'rt val{r_t_val.shape}')
            print(f'beta input val{input_beta_val_aligned.shape}')
            # Calculate the input for the factor network
            input_factor_val = calculate_factor_network_input(input_beta_val_aligned, r_t_val)

            # Ensure the shape is consistent
            assert input_beta_val_aligned.shape[0] == r_t_val.shape[0], "Shape mismatch in validation data"

            #################
            ### TEST DATA ##
            #################

            input_beta_test = np.array([c_normalized[:, :, t] for t in range(train_size + val_size, T)])  # Shift the returns forward
            input_beta_test_aligned = input_beta_test[:-2, :, :]  # Align correctly

            # Calculate forward returns for training data
            r_t_test = np.array([val_data[:, t+1] for t in range(test_size - 1)])  # Forward returns
            print(f'rt val{r_t_test.shape}')
            print(f'beta input test{input_beta_test_aligned.shape}')
            # Calculate the input for the factor network
            input_factor_test = calculate_factor_network_input(input_beta_test_aligned, r_t_test)

            # Ensure the shape is consistent
            assert input_beta_test_aligned.shape[0] == r_t_test.shape[0], "Shape mismatch in test data"

            best_val_R2_nn = -np.inf
            best_l1_value = None

            for l1_value in l1_values:
                # Build and train the neural network model with current l1_value
                main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, num_hidden_layers=num_hidden_layers, return_components=True)


                # Train the model with early stopping
                main_model.fit(
                    [input_beta_train_aligned, input_factor_train],
                    r_t_train,  # Use appropriate train data slice
                    epochs=100,  # Set a high number of epochs
                    batch_size=45,
                    verbose=0,
                    validation_data=([input_beta_val_aligned, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                    callbacks=[early_stopping]
                )

                # Evaluate on validation data
                val_pred = main_model.predict([input_beta_val_aligned, input_factor_val])
                val_R2_nn = np.mean(1 - np.var(r_t_val - val_pred, axis=1) / np.var(r_t_val, axis=1))

                # Select the best l1_value based on validation performance
                if val_R2_nn > best_val_R2_nn:
                    best_val_R2_nn = val_R2_nn
                    best_l1_value = l1_value
            print('------------------------------------------------')
            print(f'Best val R^2 {best_val_R2_nn}')

            # Train final model with the best l1_value on the entire training data
            main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, num_hidden_layers=num_hidden_layers, return_components=True)
            main_model.fit(
                [input_beta_train_aligned, input_factor_train],
                r_t_train,  # Use appropriate train data slice
                epochs=100,  # Set a high number of epochs
                batch_size=45,
                verbose=0,
                validation_data=([input_beta_val_aligned, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                callbacks=[early_stopping]
            )


            # Evaluate on test data
            test_pred = main_model.predict([input_beta_test_aligned, input_factor_test])

            # Residuals
            residuals = r_t_test - test_pred

            # RSS: Residual Sum of Squares
            rss = np.sum(residuals ** 2, axis=1)

            # TSS: Total Sum of Squares
            mean_actual = np.mean(r_t_test, axis=1, keepdims=True)
            tss = np.sum((r_t_test - mean_actual) ** 2, axis=1)

            # R^2 for each asset
            r2_values = 1 - (rss / tss)

            # Average R^2
            test_R2_nn = np.mean(r2_values)


            print(f'TEST R^2 : {test_R2_nn}')
            # Calculate predicted x_t
            predicted_x_t = calculate_factor_network_input(input_beta_test_aligned, test_pred)
            print('------------------------------------------------')

            # Evaluate predicted x_t
            true_x_t = input_factor_test
            RSS = np.sum((true_x_t - predicted_x_t) ** 2)
            TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
            R2_x_t = 1 - (RSS / TSS)
            print(f'R^2 for TEST x_t: {R2_x_t}')

            # Calculate the Sharpe ratio for predicted x_t OOS
            sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t)

            print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
            print('------------------------------------------------')

            # Calculate the portfolio return for each time period
            portfolio_returns = np.zeros(r_t_test.shape[1])
            for t in range(r_t_test.shape[1]):
                portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], r_t_test[:, t])

            # Calculate the cumulative portfolio return
            cumulative_portfolio_return = np.cumsum(portfolio_returns)

            # Create a DataFrame for the portfolio returns
            start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
            date_range = pd.date_range(start_date, periods=r_t_test.shape[1], freq='M')
            df_portfolio_returns = pd.DataFrame({
                'Date': date_range,
                'Portfolio_Return': portfolio_returns,
                'Cumulative_Portfolio_Return': cumulative_portfolio_return
            })

            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)


            print(f'Sharpe ratio for long-short portfolio returns : {long_short_portfolio_sharpe_ratio}')

            print('------------------------------------------------')
            print(f'Iteration : {i}|TEST R^2 : {test_R2_nn}')
            results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
            print('------------------------------------------------')
            if (i + 1) % 10 == 0:
                print(f'Simulation {i + 1}/{num_simulations} completed')
                print('------------------------------------------------')
            scores.append([str(num_hidden_layers) ,factor, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, test_R2_nn, best_l1_value, adjusted_R2, R2_x_t])


        # Convert scores list to DataFrame
        scores_df = pd.DataFrame(scores, columns=cols)
        scores_dict[(num_hidden_layers, factor)] = scores_df

        scores_df.to_hdf(str(results_path / f'scores_{num_hidden_layers}_{factor}.h5'), key='scores', mode='w')



# Plotting the results for each model and factor
for (num_hidden_layers, factor), df in scores_dict.items():
    plt.figure(figsize=(20, 5))  # Adjusted figure size to accommodate the extra subplot

    plt.subplot(1, 5, 1)
    plt.hist(df['Adjusted $R^2$ DGP'], bins=20, edgecolor='black')
    plt.title(f'Adjusted $R^2$ DGP, Factor {factor}')
    plt.xlabel('Adjusted $R^2$ DGP')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 2)
    plt.hist(df['$R^2$ for $r_t$'], bins=20, edgecolor='black')
    plt.title(f'$R^2$ for $r_t$: $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Test $R^2$')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 3)
    plt.hist(df['$R^2$ for $x_t$'], bins=20, edgecolor='black')
    plt.title(f'$R^2$ for $x_t$: $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('$R^2$ for $x_t$')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 4)
    plt.hist(df['$x_t$ Sharpe Ratio'], bins=20, edgecolor='black')
    plt.title(f'SR $x_t$ for $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Sharpe Ratio $x_t$')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 5)  # Added subplot for the long-short portfolio Sharpe ratio
    plt.hist(df['Long-Short Portfolio SR'], bins=20, edgecolor='black')
    plt.title(f'Long-Short Portfolio SR for $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Long-Short Portfolio SR')
    plt.ylabel('Frequency')

    plt.tight_layout()

    # Save the plot
    plot_filename = results_path / f'model_{num_hidden_layers}_factor_{factor}.png'
    plt.savefig(plot_filename)
    plt.close()
